# Nemotron 3 Ultra/Super + LangChain Integration

This notebook demonstrates how to use NVIDIA Nemotron models (via `langchain-nvidia-ai-endpoints`) for agentic AI workflows with tool-calling, structured output, and RAG.

## Prerequisites

```bash
pip install langchain-nvidia-ai-endpoints langchain-core langchain
```

You'll need an NVIDIA API key from the [NVIDIA API Catalog](https://build.nvidia.com/).

In [ ]:
import os

# Set your NVIDIA API key
# os.environ["NVIDIA_API_KEY"] = "nvapi-..."  # Replace with your key

if "NVIDIA_API_KEY" not in os.environ:
    print("⚠️  NVIDIA_API_KEY not set. Please set it to run the examples.")
else:
    print(f"✅ NVIDIA_API_KEY configured (length: {len(os.environ['NVIDIA_API_KEY'])})")

## 1. Nemotron 3 Super — Cost-Effective Agentic Model

Nemotron 3 Super (53B MoE, 8B active) is optimized for cost-effective agentic workflows with strong tool-calling and structured output.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# Nemotron 3 Super — cost-effective agentic model
llm_super = ChatNVIDIA(
    model="nvidia/nemotron-3-super-53b-a8b",
    max_tokens=4096,
    temperature=0.1,
)

response = llm_super.invoke(
    "Plan a 3-step agentic workflow for competitive research in oncology drug discovery. "
    "Return as JSON with steps, tools needed, and expected outputs."
)
print(response.content)

## 2. Nemotron 3 Ultra — Flagship Agentic Model

Nemotron 3 Ultra (120B MoE, 12B active) is NVIDIA's flagship model for agentic AI with 1M context window and best-in-class tool-calling.

In [ ]:
# Nemotron 3 Ultra — flagship agentic model
llm_ultra = ChatNVIDIA(
    model="nvidia/nemotron-3-ultra-120b-a12b",
    max_tokens=8192,
    temperature=0.1,
)

# Enable thinking mode for complex reasoning
response = llm_ultra.invoke(
    "Think step by step: Design a clinical trial protocol for a novel CAR-T therapy "
    "targeting CD19+ B-cell malignancies. Include endpoints, stratification, "
    "and statistical considerations. Show your reasoning."
)
print(response.content[:2000] + "..." if len(response.content) > 2000 else response.content)

## 3. Structured Output / JSON Schema

Nemotron models excel at structured output. Use `.with_structured_output()` for type-safe responses.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class ResearchPlan(BaseModel):
    """Structured output for a research plan."""
    objective: str = Field(description="Primary research objective")
    hypotheses: List[str] = Field(description="Testable hypotheses")
    methods: List[str] = Field(description="Methodology steps")
    expected_outcomes: List[str] = Field(description="Expected results")
    risks: List[str] = Field(description="Potential risks and mitigations")
    confidence: float = Field(description="Confidence score 0-1", ge=0, le=1)

# Get structured output
structured_llm = llm_super.with_structured_output(ResearchPlan)

plan = structured_llm.invoke(
    "Create a research plan for investigating resistance mechanisms to CD19 CAR-T therapy in DLBCL."
)

print(f"Objective: {plan.objective}")
print(f"Hypotheses: {plan.hypotheses}")
print(f"Methods: {plan.methods}")
print(f"Confidence: {plan.confidence:.2f}")

## 4. RAG with NV-Embed-QA + NV-Rerank-QA

Use NVIDIA's state-of-the-art embeddings and reranker for precision retrieval.

In [ ]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, NVIDIARerank
from langchain_core.documents import Document

# Embeddings for retrieval
embedder = NVIDIAEmbeddings(model="nvidia/nv-embed-qa")

# Sample documents
docs = [
    Document(page_content="CAR-T cell therapy targets CD19 on B-cells for B-cell malignancies.", metadata={"source": "review-1"}),
    Document(page_content="CD22-directed CAR-T shows promise for CD19-negative relapse.", metadata={"source": "review-2"}),
    Document(page_content="CRISPR-engineered T-cells with PD-1 knockout enhance persistence.", metadata={"source": "review-3"}),
]

# Embed documents
embeddings = embedder.embed_documents([d.page_content for d in docs])
print(f"Embedded {len(docs)} documents, dimension: {len(embeddings[0])}")

# Reranker for precision
reranker = NVIDIARerank(model="nvidia/nv-rerank-qa")

query = "What are emerging targets for CAR-T therapy resistance?"
reranked = reranker.compress_documents(query=query, documents=docs)

print(f"\nReranked results for: {query}")
for i, doc in enumerate(reranked):
    print(f"  {i+1}. [{doc.metadata['source']}] {doc.page_content[:80]}...")

## 5. Tool-Calling Agent with Nemotron

Nemotron models have native function calling. Build agents with tools for PubMed, DrugBank, code execution.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# Define tools
@tool
def search_pubmed(query: str, limit: int = 5) -> str:
    """Search PubMed for biomedical literature."""
    # Placeholder - integrate with NCBI E-utilities in production
    return f"PubMed results for: {query} (limit: {limit}) - [Integrate with NCBI E-utilities API]"

@tool
def fetch_drug_info(drug_name: str) -> str:
    """Fetch drug information from DrugBank."""
    # Placeholder - integrate with DrugBank API in production
    return f"Drug info for: {drug_name} - [Integrate with DrugBank API]"

# Create agent with Nemotron 3 Super
llm = ChatNVIDIA(
    model="nvidia/nemotron-3-super-53b-a8b",
    max_tokens=4096,
    temperature=0.1,
)

tools = [search_pubmed, fetch_drug_info]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a biomedical research agent. Use tools to find evidence. Cite sources."),
    ("user", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Example query
# result = agent_executor.invoke({"input": "Find recent clinical trials for CD19 CAR-T in relapsed/refractory DLBCL"})
# print(result["output"])

print("Agent configured. Uncomment the invoke call to run (requires NVIDIA_API_KEY).")

## 6. Self-Hosted NIM (Local-First, Zero API Cost)

For sovereign deployments with PHI/private data, deploy Nemotron as a self-hosted NIM with TensorRT-LLM optimization.

In [ ]:
import os

# Local NIM endpoint (update for your deployment)
NIM_BASE_URL = os.getenv("NIM_BASE_URL", "http://localhost:8000/v1")
NIM_MODEL = os.getenv("NIM_MODEL", "nvidia/nemotron-3-ultra-120b-a12b")

# Example: Using local NIM with ChatNVIDIA
# llm_local = ChatNVIDIA(
#     model=NIM_MODEL,
#     base_url=NIM_BASE_URL,
#     api_key="dummy",  # Not needed for local NIM
#     max_tokens=4096,
# )

print(f"Local NIM endpoint: {NIM_BASE_URL}")
print(f"Model: {NIM_MODEL}")
print("\nTo deploy local NIM:")
print("1. Obtain NVIDIA AI Enterprise license")
print("2. Pull NIM container: docker pull nvcr.io/nim/nvidia/nemotron-3-ultra:latest")
print("3. Run: docker run -d --gpus all -p 8000:8000 nvcr.io/nim/nvidia/nemotron-3-ultra:latest")
print("4. Set NIM_BASE_URL=http://localhost:8000/v1 and NIM_MODEL=nvidia/nemotron-3-ultra-120b-a12b")
print("\nBenefits:")
print("- Zero API costs (compute only)")
print("- Full data sovereignty (PHI never leaves infrastructure)")
print("- TensorRT-LLM optimized inference")
print("- 1M context window")

## 7. Cost Estimation

Estimate costs before running workloads.

In [ ]:
# NVIDIA API pricing (per 1M tokens, approximate as of 2026)
NVIDIA_PRICING = {
    "nvidia/nemotron-3-ultra-120b-a12b": {"input": 5.00, "output": 15.00},
    "nvidia/nemotron-3-super-53b-a8b": {"input": 2.00, "output": 6.00},
    "nvidia/nv-embed-qa": {"input": 0.10, "output": 0.0},
    "nvidia/nv-rerank-qa": {"input": 1.00, "output": 0.0},
}

# Comparison baselines
GPT55 = {"input": 10.0, "output": 30.0}
OPUS47 = {"input": 15.0, "output": 75.0}

def estimate_cost(model_id: str, input_tokens: int, output_tokens: int) -> dict:
    pricing = NVIDIA_PRICING.get(model_id, {"input": 2.0, "output": 6.0})
    input_cost = (input_tokens / 1_000_000) * pricing["input"]
    output_cost = (output_tokens / 1_000_000) * pricing["output"]
    total = input_cost + output_cost
    
    gpt55 = (input_tokens / 1_000_000) * GPT55["input"] + (output_tokens / 1_000_000) * GPT55["output"]
    opus47 = (input_tokens / 1_000_000) * OPUS47["input"] + (output_tokens / 1_000_000) * OPUS47["output"]
    
    return {
        "model": model_id,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cost_usd": round(total, 6),
        "savings_vs_gpt55": round(gpt55 - total, 6),
        "savings_vs_opus47": round(opus47 - total, 6),
    }

# Compare models for a typical workload
models = [
    "nvidia/nemotron-3-ultra-120b-a12b",
    "nvidia/nemotron-3-super-53b-a8b",
]

input_tokens = 10_000
output_tokens = 2_000

print(f"{'Model':<45} {'Cost':>10} {'vs GPT-5.5':>12} {'vs Opus 4.7':>12}")
print("-" * 85)

for model in models:
    cost = estimate_cost(model, input_tokens, output_tokens)
    print(f"{cost['model']:<45} ${cost['cost_usd']:>9.4f} ${cost['savings_vs_gpt55']:>+11.4f} ${cost['savings_vs_opus47']:>+11.4f}")

## 8. Thinking Mode

Enable step-by-step reasoning for complex problems.

In [ ]:
# Enable thinking mode via system prompt
llm_thinking = ChatNVIDIA(
    model="nvidia/nemotron-3-ultra-120b-a12b",
    max_tokens=8192,
    temperature=0.1,
)

# Prepend thinking instruction
thinking_prompt = """Think step by step. Show your reasoning before providing the final answer.

Question: What are the key mechanisms of resistance to CD19 CAR-T therapy in DLBCL, """

response = llm_ultra.invoke(thinking_prompt)
print(response.content[:2000] + "..." if len(response.content) > 2000 else response.content)

## Summary

This integration provides:

1. **Nemotron 3 Ultra/Super** — NVIDIA's flagship agentic models via `ChatNVIDIA`
2. **Tool Calling** — Native function calling for PubMed, DrugBank, code execution
3. **Structured Output** — Pydantic models for type-safe responses
4. **RAG Pipeline** — NV-Embed-QA + NV-Rerank-QA for precision retrieval
5. **Self-Hosted NIM** — Zero API cost, full data sovereignty, TensorRT-LLM optimization
6. **Cost Efficiency** — 60-85% savings vs. proprietary frontiers
6. **Thinking Mode** — Step-by-step reasoning for complex problems

All components work together to make NVIDIA Nemotron the backbone of sovereign, verifiable, cost-transparent agentic AI for biomedical research.